# Trans-environment comparison (human vs chimp)

Comparison of TF expression between human and chimpanzee (parental CNCCs and an external bone dataset) to assess *trans*-environment similarity.

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Paths (EDIT): inputs read / outputs written by this notebook ---
CNCC_ASE_PATH = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Hybrids/human_chimp/iPSCs_and_CNCCs/iPSCs_CNCCs_ASE.xlsx'  # CNCC/iPSC ASE expression table
JASPAR_META_PATH = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Sequence/Motifs_and_TFBS/JASPAR2024/JASPAR2024_CORE_vertebrates_non-redundant_meta.tsv'  # JASPAR2024 TF metadata
BONE_DE_PATH = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Bone/Human/RNA-seq/In_vitro_osteogenic_cells_DE_housman.xlsx'  # Housman osteogenic DE table
TF_MERGED_PATH = '/home/labs/davidgo/Collaboration/humanMPRA/TF_analysis/final/results/hMPRA_PBM_oligo_TF_merged.tsv'  # merged hMPRA/PBM oligo TF data
output_path = '.'  # where plots/tables are written


In [ ]:
# Modular scatter correlation plot for any two columns in Yamada_TPM
def plot_scatter_correlation(
    df,
    x_col,
    y_col,
    pseudocount=0.1,
    log_scale=True,
    alpha=0.5,
    s=12,
    color="#227C9D",
    title=None,
    save_path=None,
):
    plot_df = df[[x_col, y_col]].copy()
    plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
    plot_df[y_col] = pd.to_numeric(plot_df[y_col], errors="coerce")
    plot_df = plot_df.dropna()

    x = plot_df[x_col].values + pseudocount
    y = plot_df[y_col].values + pseudocount

    pearson_r, pearson_p = pearsonr(np.log10(x), np.log10(y)) if log_scale else pearsonr(x, y)
    spearman_r, spearman_p = spearmanr(x, y)

    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Calculate 2D density for coloring
    from matplotlib.colors import Normalize
    from scipy.stats import gaussian_kde
    
    # Calculate density on log-transformed data if applicable
    if log_scale:
        x_for_kde = np.log10(x)
        y_for_kde = np.log10(y)
    else:
        x_for_kde = x
        y_for_kde = y
    
    xy = np.vstack([x_for_kde, y_for_kde])
    kde = gaussian_kde(xy)
    density = kde(xy)
    
    # Cap density at 1
    #density = np.minimum(density, 1)
    
    # Normalize density to [0, 1] for colormap
    norm = Normalize(vmin=density.min(), vmax=density.max())
    colors = custom_cmap_bolder(norm(density))
    
    ax.scatter(x, y, alpha=alpha, s=s, c=colors, edgecolors="none")
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=custom_cmap_bolder, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Density', fontsize=FONT_SIZE_small - 2)

    if log_scale:
        ax.set_xscale("log")
        ax.set_yscale("log")

    min_val = min(x.min(), y.min())
    max_val = max(x.max(), y.max())
    ax.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1.2, alpha=0.6)

    ax.set_xlabel(x_col, fontsize=FONT_SIZE_small-2)
    ax.set_ylabel(y_col, fontsize=FONT_SIZE_small-2)
    #ax.set_title(title if title else f"{x_col} vs {y_col}", fontsize=FONT_SIZE_small)
    #ax.grid(True, alpha=0.25)

    # Format p-values, showing underflow when p-value is too small
    import sys
    pearson_p_str = f"p={pearson_p:.2e}" if pearson_p > 0 else f"p < {sys.float_info.min:.2e}"
    spearman_p_str = f"p={spearman_p:.2e}" if spearman_p > 0 else f"p < {sys.float_info.min:.2e}"
    
    textstr = (
        f"n = {len(plot_df):,}\n"
        f"Pearson's r = {pearson_r:.3f} ({pearson_p_str})\n"
        f"Spearman's ρ = {spearman_r:.3f} ({spearman_p_str})"
    )
    ax.text(
        0.03, 0.97, textstr,
        transform=ax.transAxes,
        va="top",
        fontsize=FONT_SIZE_small - 6,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
    )

    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.show()

    return {"n": len(plot_df), "pearson_r": pearson_r, "pearson_p": pearson_p, "spearman_r": spearman_r, "spearman_p": spearman_p}



# Comparing the trans environemnt of chimp and human using the parental CNCCs

i want you to read the excel file that is located in /home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Hybrids/human_chimp/iPSCs_and_CNCCs/iPSCs_CNCCs_ASE.xlsx. Specifically, read the sheet called SUMMARY and skip the first three rows. Finally, save it into a df called CNCC_parental which will contain only the following cols: "FPKM - ((reads/libSize)/CDS)*10^6 - CNCC - Parental human" "FPKM - ((reads/libSize)/CDS)*10^6 - CNCC - Parental chimp" and "gene"

In [ ]:
cncc_path = CNCC_ASE_PATH

cncc_cols = [
    "FPKM - ((reads/libSize)/CDS)*10^6 - CNCC - Parental human",
    "FPKM - ((reads/libSize)/CDS)*10^6 - CNCC - Parental chimp",
    "gene",
]

CNCC_parental = pd.read_excel(
    cncc_path,
    sheet_name="SUMMARY",
    skiprows=3,
    usecols=cncc_cols
)

CNCC_parental.rename(columns={
    "FPKM - ((reads/libSize)/CDS)*10^6 - CNCC - Parental human": "Parental_Human_FPKM",
    "FPKM - ((reads/libSize)/CDS)*10^6 - CNCC - Parental chimp": "Parental_Chimp_FPKM",
    "gene": "Gene"
}, inplace=True)




### Filter for TFs that are in the JASPAR database

In [ ]:
JASPAR_dataset = pd.read_csv(JASPAR_META_PATH,
                     sep = '\t',
                       #usecols=range(0, 34), 
                     header=0)
JASPAR_TFs = (
  JASPAR_dataset["TF"]
  .dropna()
  .astype(str)
  .str.upper()
  .str.split(r"::|-", regex=True)  # split on both "::" and "-"
  .explode()
  .str.strip()
  .replace("", np.nan)
  .dropna()
  .unique()
)

print(f"Number of CNCCs before TF filtering : {len(CNCC_parental)}")
CNCC_parental_TFs = CNCC_parental[CNCC_parental['Gene'].isin(JASPAR_TFs)].copy()
print(f"Number of TFs in CNCC parental dataset: {len(CNCC_parental_TFs)}")

In [ ]:
corr_stats = plot_scatter_correlation(
    CNCC_parental,
    x_col='Parental_Human_FPKM',
    y_col='Parental_Chimp_FPKM',
    #title="Correlation between ExpLBM PRRX1 High#1 and 3DCI Step 3: 6wks #1",
    save_path=output_path + "CNCC_Parental_Human_vs_Chimp_scatter.png",
)

print(corr_stats)


In [ ]:
corr_stats = plot_scatter_correlation(
    CNCC_parental_TFs,
    x_col='Parental_Human_FPKM',
    y_col='Parental_Chimp_FPKM',
    #title="Correlation between ExpLBM PRRX1 High#1 and 3DCI Step 3: 6wks #1",
    save_path=output_path + "CNCC_Parental_Human_vs_Chimp_scatter.png",
)

print(corr_stats)



## Comparing human and Chimp trans environment using (Housman et al paper)
https://journals.plos.org/plosgenetics/article?id=10.1371/journal.pgen.1010073#sec025

In [ ]:
bone_excel_path = BONE_DE_PATH
bone_df = pd.read_excel(bone_excel_path)

print(f"Before filtering : {len(bone_df)}")
bone_df = bone_df[bone_df['cell.subset']=='Osteogenic.c1']
print(f"After filtering : {len(bone_df)}")

In [ ]:
tf_bone_df = bone_df[bone_df['Gene'].isin(JASPAR_TFs)].copy()

In [ ]:
Housman_diff_active_tfs = (
    tf_bone_df["Gene"]
    .dropna()
    .astype(str)
    .str.upper()
    .unique()
)

print(f"Number of unique TF genes: {len(Housman_diff_active_tfs)}")
print(Housman_diff_active_tfs)

### Load TF data

In [ ]:
TF_data_path = TF_MERGED_PATH

PBM_data = pd.read_csv(TF_data_path, sep='\t')

In [ ]:
PBM_data['motif_id_clean'] = PBM_data['motif_id_clean'].str.upper()

In [ ]:
bone_PBM_TF_data = PBM_data[PBM_data['motif_id_clean'].isin(Housman_diff_active_tfs)].copy()

In [ ]:
bone_PBM_TF_data